In [1]:
#!pip install mlflow

In [2]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
import os

In [3]:
# ─────────────────────────────────────────
# 1. Setting MLflow Enviroment
# ─────────────────────────────────────────
EXPERIMENT_NAME = "Iris_Classification_Experiment"
mlflow.set_tracking_uri("sqlite:///mlflow.db")  # حفظ محلي في قاعدة بيانات
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='file:e:/Academic/Work/DEBI/CLS/OneDrive_3_10-24-2024/9- Ml-Flow/mlruns/1', creation_time=1778247825848, experiment_id='1', last_update_time=1778247825848, lifecycle_stage='active', name='Iris_Classification_Experiment', tags={}, trace_location=None, workspace='default'>

In [4]:

def load_and_prepare_data():
    iris = load_iris()
    X = pd.DataFrame(iris.data, columns=iris.feature_names)
    y = pd.Series(iris.target, name="species")
    
    print(f" Dataset loaded: {X.shape[0]} samples, {X.shape[1]} features")
    print(f"   Classes: {iris.target_names.tolist()}")
    print(f"   Class distribution:\n{y.value_counts().to_string()}\n")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, iris.target_names, scaler



In [5]:
# ─────────────────────────────────────────
# 3. saving Confusion Matrix as a pic
# ─────────────────────────────────────────
def save_confusion_matrix(y_test, y_pred, class_names, model_name):
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"Confusion Matrix - {model_name}", fontsize=13, fontweight="bold")
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")
    plt.tight_layout()
    
    os.makedirs("artifacts", exist_ok=True)
    path = f"artifacts/cm_{model_name.replace(' ', '_')}.png"
    fig.savefig(path, dpi=150)
    plt.close()
    return path

In [6]:
# ─────────────────────────────────────────
# 4. Training Functions MLflow Logging
# ─────────────────────────────────────────
def train_and_log(model, model_name, params, X_train, X_test, y_train, y_test, class_names):
    print(f" Training: {model_name}")
    
    with mlflow.start_run(run_name=model_name):
        
        # --- training models---
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # --- calculation Metrics ---
        acc       = accuracy_score(y_test, y_pred)
        f1        = f1_score(y_test, y_pred, average="weighted")
        precision = precision_score(y_test, y_pred, average="weighted")
        recall    = recall_score(y_test, y_pred, average="weighted")
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
        cv_mean   = cv_scores.mean()
        cv_std    = cv_scores.std()

        # --- Log Parameters ---
        mlflow.log_params(params)
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("test_size", 0.2)
        mlflow.log_param("random_state", 42)

        # --- Log Metrics ---
        mlflow.log_metric("accuracy",       acc)
        mlflow.log_metric("f1_score",       f1)
        mlflow.log_metric("precision",      precision)
        mlflow.log_metric("recall",         recall)
        mlflow.log_metric("cv_mean",        cv_mean)
        mlflow.log_metric("cv_std",         cv_std)

        # --- Log Confusion Matrix Artifact ---
        cm_path = save_confusion_matrix(y_test, y_pred, class_names, model_name)
        mlflow.log_artifact(cm_path)

        # --- Log Classification Report as text ---
        report = classification_report(y_test, y_pred, target_names=class_names)
        os.makedirs("artifacts", exist_ok=True)
        report_path = f"artifacts/report_{model_name.replace(' ', '_')}.txt"
        with open(report_path, "w") as f:
            f.write(f"Model: {model_name}\n\n")
            f.write(report)
        mlflow.log_artifact(report_path)

        # --- Log for the model ---
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            registered_model_name=f"iris_{model_name.replace(' ', '_')}"
        )

        run_id = mlflow.active_run().info.run_id
        
        print(f"    Accuracy:  {acc:.4f}")
        print(f"    F1 Score:  {f1:.4f}")
        print(f"    CV Mean:   {cv_mean:.4f} ± {cv_std:.4f}")
        print(f"    Run ID:   {run_id}\n")
        
        return {
            "model_name": model_name,
            "run_id":     run_id,
            "accuracy":   acc,
            "f1_score":   f1,
            "precision":  precision,
            "recall":     recall,
            "cv_mean":    cv_mean,
        }


In [ ]:
# ─────────────────────────────────────────
# 5. Comparing the results and Saving it.
# ─────────────────────────────────────────
def compare_and_save(results):
    df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
    
    print("\n" + "="*65)
    print(" RESULTS COMPARISON")
    print("="*65)
    print(df[["model_name", "accuracy", "f1_score", "cv_mean"]].to_string(index=False))
    
    best = df.iloc[0]
    print(f"\n Best Model: {best['model_name']}")
    print(f"   Accuracy: {best['accuracy']:.4f} | F1: {best['f1_score']:.4f}")
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    metrics  = ["accuracy", "f1_score", "precision", "recall"]
    colors   = ["#4C72B0", "#DD8452", "#55A868"]
    
    x = np.arange(len(metrics))
    width = 0.25
    for i, (_, row) in enumerate(df.iterrows()):
        vals = [row[m] for m in metrics]
        axes[0].bar(x + i * width, vals, width, label=row["model_name"], color=colors[i])
    
    axes[0].set_xticks(x + width)
    axes[0].set_xticklabels(metrics, rotation=15)
    axes[0].set_ylim(0.8, 1.02)
    axes[0].set_title("Model Metrics Comparison", fontweight="bold")
    axes[0].legend()
    axes[0].set_ylabel("Score")
    
    axes[1].barh(df["model_name"], df["cv_mean"], color=colors[:len(df)],
                 xerr=df["cv_mean"].std())
    axes[1].set_title("Cross-Validation Mean Accuracy", fontweight="bold")
    axes[1].set_xlim(0.8, 1.02)
    axes[1].set_xlabel("CV Accuracy")
    
    plt.tight_layout()
    comp_path = "artifacts/comparison.png"
    fig.savefig(comp_path, dpi=150)
    plt.close()
    print(f"\n Comparison chart saved → {comp_path}")
    
    csv_path = "artifacts/results_summary.csv"
    df.to_csv(csv_path, index=False)
    print(f" Results CSV saved  → {csv_path}")
    
    return df

In [8]:
# ─────────────────────────────────────────
# 6. Main
# ─────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 55)
    print("    MLflow Classification Project - Iris Dataset")
    print("=" * 55 + "\n")

    X_train, X_test, y_train, y_test, class_names, scaler = load_and_prepare_data()

    # define the model and the params
    experiments = [
        (
            LogisticRegression(C=1.0, max_iter=200, random_state=42),
            "Logistic Regression",
            {"C": 1.0, "max_iter": 200, "solver": "lbfgs"}
        ),
        (
            RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
            "Random Forest",
            {"n_estimators": 100, "max_depth": 5, "criterion": "gini"}
        ),
        (
            SVC(C=1.0, kernel="rbf", probability=True, random_state=42),
            "SVM RBF",
            {"C": 1.0, "kernel": "rbf", "gamma": "scale"}
        ),
    ]

    results = []
    for model, name, params in experiments:
        result = train_and_log(
            model, name, params,
            X_train, X_test, y_train, y_test, class_names
        )
        results.append(result)

    compare_and_save(results)

    print("\n" + "=" * 55)
    print(" All runs logged to MLflow!")
    print("\n▶ To view the UI, run:")
    print("   mlflow ui --backend-store-uri sqlite:///mlflow.db")
    print("   Then open: http://127.0.0.1:5000")
    print("=" * 55)


    MLflow Classification Project - Iris Dataset

 Dataset loaded: 150 samples, 4 features
   Classes: ['setosa', 'versicolor', 'virginica']
   Class distribution:
species
0    50
1    50
2    50

 Training: Logistic Regression


2026/05/15 14:19:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 14:19:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'iris_Logistic_Regression' already exists. Creating a new version of this model...
Created version '3' of model 'iris_Logistic_Regression'.


    Accuracy:  0.9333
    F1 Score:  0.9333
    CV Mean:   0.9583 ± 0.0264
    Run ID:   d3c664c633b5459e8230d25fc507dc8b

 Training: Random Forest


2026/05/15 14:19:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 14:19:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'iris_Random_Forest' already exists. Creating a new version of this model...
Created version '3' of model 'iris_Random_Forest'.


    Accuracy:  0.9333
    F1 Score:  0.9333
    CV Mean:   0.9500 ± 0.0167
    Run ID:   8b69dba494ee4b90b143db9f47d4da21

 Training: SVM RBF


2026/05/15 14:20:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/15 14:20:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'iris_SVM_RBF' already exists. Creating a new version of this model...
Created version '3' of model 'iris_SVM_RBF'.


    Accuracy:  0.9667
    F1 Score:  0.9666
    CV Mean:   0.9667 ± 0.0312
    Run ID:   8e001d4573cb431e92d5f9c61efa8878


 RESULTS COMPARISON
         model_name  accuracy  f1_score  cv_mean
            SVM RBF  0.966667  0.966583 0.966667
Logistic Regression  0.933333  0.933333 0.958333
      Random Forest  0.933333  0.933333 0.950000

 Best Model: SVM RBF
   Accuracy: 0.9667 | F1: 0.9666

 Comparison chart saved → artifacts/comparison.png
 Results CSV saved  → artifacts/results_summary.csv

 All runs logged to MLflow!

▶ To view the UI, run:
   mlflow ui --backend-store-uri sqlite:///mlflow.db
   Then open: http://127.0.0.1:5000


In [9]:
!mlflow ui --backend-store-uri sqlite:///mlflow.db

^C
